# Agents as Tools with Strands Agents


"Agents as Tools" is an architectural pattern in AI systems where specialized AI agents are wrapped as callable functions (tools) that can be used by other agents. This creates a hierarchical structure where:

1. A primary "orchestrator" agent handles user interaction and determines which specialized agent to call

2. Specialized "tool agents" perform domain-specific tasks when called by the orchestrator

This approach mimics human team dynamics, where a manager coordinates specialists, each bringing unique expertise to solve complex problems. Rather than a single agent trying to handle everything, tasks are delegated to the most appropriate specialized agent.



## Key Benefits and Core Principles

The "Agents as Tools" pattern offers several advantages:

- Separation of concerns: Each agent has a focused area of responsibility, making the system easier to understand and maintain
- Hierarchical delegation: The orchestrator decides which specialist to invoke, creating a clear chain of command
- Modular architecture: Specialists can be added, removed, or modified independently without affecting the entire system
- Improved performance: Each agent can have tailored system prompts and tools optimized for its specific task


In [1]:
!pip install -r requirements.txt

In [2]:
import os

from strands import Agent, tool
from strands_tools import file_write


In this module we will be creating an orchestrator based multi-agent workflow. 

<div style="text-align:left">
    <img src="images/architecture.png" width="75%" />
</div>

We will also explore `use_llm` which allows use to create nested agents.

## Research Agent

Lets first create a basic reasearch assistant with http_request tool. 

In [3]:
RESEARCH_ASSISTANT_PROMPT = """You are a specialized research assistant. Focus only on providing
factual, well-sourced information in response to research questions.
Always cite your sources when possible."""

In [4]:
research_agent = Agent(
    model="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    system_prompt=RESEARCH_ASSISTANT_PROMPT,
    # tools=[http_request]  # Here you can enable an agentic ai search tool
)

query = "Overview of Amazon Bedrock and its features"
# Call the agent and return its response
response = research_agent(query)

# Amazon Bedrock Overview

Amazon Bedrock is a fully managed service launched by AWS in 2023 that provides access to foundation models (FMs) from leading AI companies through a single API.

## Key Features

### 1. **Multiple Foundation Models**
- Access to models from **Anthropic (Claude)**, **AI21 Labs**, **Cohere**, **Meta (Llama)**, **Stability AI**, and **Amazon Titan**
- Ability to choose models based on specific use case requirements
- Support for text, image, and embedding generation

### 2. **Model Customization**
- **Fine-tuning**: Customize models with your own labeled datasets
- **Continued pre-training**: Adapt models to specific domains with unlabeled data
- Private customization - your data remains secure and isn't used to train base models

### 3. **Retrieval-Augmented Generation (RAG)**
- **Knowledge Bases**: Connect FMs to your proprietary data sources
- Integrates with vector databases for contextual information retrieval
- Improves accuracy and reduces hallucinations

Now we can wrap this agent as a tool. Allowing other agents to interact with it. 

####  Best Practices for Agent as Tools

When implementing the "Agents as Tools" pattern with Strands Agents:

1. Clear tool documentation: Write descriptive docstrings that explain the agent's expertise
2. Focused system prompts: Keep each specialized agent tightly focused on its domain
3. Proper response handling: Use consistent patterns to extract and format responses
4. Tool selection guidance: Give the orchestrator clear criteria for when to use each specialized agen

In [5]:
@tool
def research_assistant(query: str) -> str:
    """
    Process and respond to research-related queries.

    Args:
        query: A research question requiring factual information

    Returns:
        A detailed research answer with citations
    """
    try:
        # Strands agents makes it easy to create a specialized agent
        research_agent = Agent(
            model="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
            system_prompt=RESEARCH_ASSISTANT_PROMPT,
        )

        # Call the agent and return its response
        response = research_agent(query)
        return str(response)
    except Exception as e:
        return f"Error in research assistant: {str(e)}"

Now lets follow the best practices and create `product_recommendation_assistant`, `trip_planning_assistant`, and `orchestrator` agent.

### Product Recommendation Assistant

In [6]:
@tool
def product_recommendation_assistant(query: str) -> str:
    """
    Handle product recommendation queries by suggesting appropriate products.

    Args:
        query: A product inquiry with user preferences

    Returns:
        Personalized product recommendations with reasoning
    """
    try:
        product_agent = Agent(
            model="us.anthropic.claude-sonnet-4-5-20250929-v1:0",  # Optional: Specify the model ID
            system_prompt="""You are a specialized product recommendation assistant.
            Provide personalized product suggestions based on user preferences. Always cite your sources.""",
        )
        # Call the agent and return its response
        response = product_agent(query)

        return str(response)
    except Exception as e:
        return f"Error in product recommendation: {str(e)}"

In [7]:
product_recommendation_assistant("Product recommendations for flying cars")

# Flying Car Product Recommendations

I need to clarify that **fully autonomous flying cars for consumer purchase are not yet commercially available** as of my last update. However, here are the closest alternatives and emerging options:

## 1. **Currently Available/Pre-Order**

### **PAL-V Liberty** (Netherlands)
- **Type**: Gyroplane/car hybrid
- **Price**: ~$400,000 - $600,000
- **Status**: Limited production, requires pilot license
- **Features**: Road-legal car that converts to gyrocopter; 2-seater

### **Samson Sky Switchblade** (USA)
- **Price**: ~$170,000 (reservation holders)
- **Status**: Pre-orders accepted, production pending
- **Features**: 3-wheel vehicle with foldable wings

## 2. **Electric VTOL (Near Future)**

### **Joby Aviation eVTOL**
- **Expected**: 2025+ (air taxi service first)
- **Type**: Electric vertical takeoff aircraft
- **Note**: Initially commercial service, not personal ownership

### **Lilium Jet**
- **Type**: Electric jet-powered VTOL
- **Status**: In 

"# Flying Car Product Recommendations\n\nI need to clarify that **fully autonomous flying cars for consumer purchase are not yet commercially available** as of my last update. However, here are the closest alternatives and emerging options:\n\n## 1. **Currently Available/Pre-Order**\n\n### **PAL-V Liberty** (Netherlands)\n- **Type**: Gyroplane/car hybrid\n- **Price**: ~$400,000 - $600,000\n- **Status**: Limited production, requires pilot license\n- **Features**: Road-legal car that converts to gyrocopter; 2-seater\n\n### **Samson Sky Switchblade** (USA)\n- **Price**: ~$170,000 (reservation holders)\n- **Status**: Pre-orders accepted, production pending\n- **Features**: 3-wheel vehicle with foldable wings\n\n## 2. **Electric VTOL (Near Future)**\n\n### **Joby Aviation eVTOL**\n- **Expected**: 2025+ (air taxi service first)\n- **Type**: Electric vertical takeoff aircraft\n- **Note**: Initially commercial service, not personal ownership\n\n### **Lilium Jet**\n- **Type**: Electric jet-powe

### Trip Planning Assistant

In [8]:
@tool
def trip_planning_assistant(query: str) -> str:
    """
    Create travel itineraries and provide travel advice.

    Args:
        query: A travel planning request with destination and preferences

    Returns:
        A detailed travel itinerary or travel advice
    """
    try:
        travel_agent = Agent(
            model="us.anthropic.claude-sonnet-4-5-20250929-v1:0",  # Optional: Specify the model ID
            system_prompt="""You are a specialized travel planning assistant.
            Create detailed travel itineraries based on user preferences.""",
        )
        # Call the agent and return its response
        response = travel_agent(query)

        return str(response)
    except Exception as e:
        return f"Error in trip planning: {str(e)}"

### Orchestrator Agent

In [9]:
# Define orchestrator system prompt with clear tool selection guidance
MAIN_SYSTEM_PROMPT = """
You are an assistant that routes queries to specialized agents:
- For research questions and factual information → Use the research_assistant tool
- For product recommendations and shopping advice → Use the product_recommendation_assistant tool
- For travel planning and itineraries → Use the trip_planning_assistant tool
- For simple questions not requiring specialized knowledge → Answer directly

Always select the most appropriate tool based on the user's query.
"""

In [10]:
# Strands Agents allows easy integration of agent tools
orchestrator = Agent(
    model="us.anthropic.claude-sonnet-4-5-20250929-v1:0",  # Optional: Specify the model ID
    system_prompt=MAIN_SYSTEM_PROMPT,
    tools=[
        research_assistant,
        product_recommendation_assistant,
        trip_planning_assistant,
        file_write,
    ],
)

In [11]:
# Example: E-commerce Customer Service System
customer_query = (
    "I'm looking for hiking boots. Write the final response to current directory."
)

os.environ["BYPASS_TOOL_CONSENT"] = "true"

# The orchestrator automatically determines this requires multiple specialized agents
response = orchestrator(customer_query)

I'll help you with hiking boot recommendations and save the response to a file.
Tool #1: product_recommendation_assistant
I'd be happy to help you find hiking boots! To give you the best recommendations, I need to know a bit more about your needs:

1. **What type of hiking** are you planning?
   - Day hikes on maintained trails
   - Backpacking with heavy loads
   - Technical/mountaineering
   - Light trail walking

2. **Terrain conditions**?
   - Dry/rocky trails
   - Wet/muddy conditions
   - Snow/winter hiking
   - Mixed terrain

3. **Your budget range**?
   - Under $100
   - $100-$200
   - $200-$300
   - $300+

4. **Any specific preferences**?
   - Ankle support level (low, mid, or high cut)
   - Waterproofing needs
   - Weight preference (lightweight vs. durability)
   - Fit concerns (wide feet, narrow heel, etc.)

5. **Experience level**?
   - Beginner
   - Intermediate
   - Advanced hiker

Once you share these details, I can provide specific product recommendations with features

╔═════════ File Write Operation ═════════╗
║                                        ║
║ Path: hiking_boots_recommendations.txt ║
║ Size: 950 characters                   ║
║                                        ║
╚════════════════════════════════════════╝

╔══════════════════════ Write Successful ═══════════════════════╗
║ File written successfully to hiking_boots_recommendations.txt ║
╚═══════════════════════════════════════════════════════════════╝

Done! I've saved the hiking boots recommendation guide to `hiking_boots_recommendations.txt` in the current directory. 

The file contains questions to help narrow down the best hiking boots for your needs. If you'd like me to provide specific product recommendations, please share details about:
- The type of hiking you'll be doing
- Your terrain and weather conditions
- Your budget
- Any specific preferences (waterproofing, ankle support, etc.)

Then I can give you detailed recommendations with specific boot models!

Lets look at the messages of the orchestrator. Here you can see the agent decided to use the sub-agent as tool

In [12]:
orchestrator.messages

[{'role': 'user',
  'content': [{'text': "I'm looking for hiking boots. Write the final response to current directory."}]},
 {'role': 'assistant',
  'content': [{'text': "I'll help you with hiking boot recommendations and save the response to a file."},
   {'toolUse': {'toolUseId': 'tooluse_YOCwyVPhlkzTxt2N7AupwS',
     'name': 'product_recommendation_assistant',
     'input': {'query': "I'm looking for hiking boots"}}}]},
 {'role': 'user',
  'content': [{'toolResult': {'toolUseId': 'tooluse_YOCwyVPhlkzTxt2N7AupwS',
     'status': 'success',
     'content': [{'text': "I'd be happy to help you find hiking boots! To give you the best recommendations, I need to know a bit more about your needs:\n\n1. **What type of hiking** are you planning?\n   - Day hikes on maintained trails\n   - Backpacking with heavy loads\n   - Technical/mountaineering\n   - Light trail walking\n\n2. **Terrain conditions**?\n   - Dry/rocky trails\n   - Wet/muddy conditions\n   - Snow/winter hiking\n   - Mixed terra

In [13]:
customer_query = "Can you help me plan my trip to Patagonia"

response = orchestrator(customer_query)


Tool #3: trip_planning_assistant
# Patagonia Travel Itinerary Planning

I'd be happy to help you plan an amazing trip to Patagonia! To create the perfect itinerary for you, I need to understand your preferences better.

## Key Questions:

**1. Duration & Timing**
- How many days do you have for this trip?
- What time of year are you planning to visit?
  - *Best seasons: Oct-Apr (spring/summer), fewer crowds in shoulder seasons*

**2. Which Side (or Both)?**
- **Argentine Patagonia** (El Calafate, El Chaltén, Peninsula Valdés)
- **Chilean Patagonia** (Torres del Paine, Punta Arenas)
- Both sides?

**3. Travel Style & Interests**
- Activity level: Relaxed sightseeing, moderate hiking, or challenging trekking?
- Interests: Glaciers, wildlife, mountains, photography, culture?
- Accommodation preference: Budget hostels, mid-range, or luxury lodges?

**4. Budget Range**
- Budget-conscious, moderate, or luxury travel?

**5. Special Requirements**
- Any physical limitations?
- Traveling solo,

In [14]:
orchestrator.messages

[{'role': 'user',
  'content': [{'text': "I'm looking for hiking boots. Write the final response to current directory."}]},
 {'role': 'assistant',
  'content': [{'text': "I'll help you with hiking boot recommendations and save the response to a file."},
   {'toolUse': {'toolUseId': 'tooluse_YOCwyVPhlkzTxt2N7AupwS',
     'name': 'product_recommendation_assistant',
     'input': {'query': "I'm looking for hiking boots"}}}]},
 {'role': 'user',
  'content': [{'toolResult': {'toolUseId': 'tooluse_YOCwyVPhlkzTxt2N7AupwS',
     'status': 'success',
     'content': [{'text': "I'd be happy to help you find hiking boots! To give you the best recommendations, I need to know a bit more about your needs:\n\n1. **What type of hiking** are you planning?\n   - Day hikes on maintained trails\n   - Backpacking with heavy loads\n   - Technical/mountaineering\n   - Light trail walking\n\n2. **Terrain conditions**?\n   - Dry/rocky trails\n   - Wet/muddy conditions\n   - Snow/winter hiking\n   - Mixed terra

### Calling multiple agents 

In [15]:
orchestrator.messages = []

In [ ]:
query = "Can you do a research on spain? Also help me plan a 7 day trip."

orchestrator(query)

Behind the scenes, the orchestrator will:
1. First call the `research_assistant`
2. Then call `trip_planning_assistant`
3. Combine these specialized responses into a cohesive answer that addresses both the queries

### Sequential Agent Communication Pattern


The agent tool can also combine multiple agents together. In this example we will provide output of `research_agent` to `summary_agent` and return the summarized response.

In [16]:
 # define the user query
topic = "generative Ai"
# Create a research agent
research_agent = Agent(
    model="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    system_prompt=RESEARCH_ASSISTANT_PROMPT,
)
# Create a summarization agent
summary_agent = Agent(
    model="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    system_prompt="""
    You are a summarization specialist focused on distilling complex information into clear, concise summaries.
    Your primary goal is to take detailed information and extract the key points, main arguments, and critical data.
    You should maintain the accuracy of the original content while making it more digestible.
    Focus on clarity, brevity, and highlighting the most important aspects of the information.
    """,
)

print("Multiple agents created successfully!")
print(f"\n🔍 RESEARCH AGENT working on: {topic}\n") 
try:
    # Agent 1: Invoke research agent
    research_response = research_agent(
        f"Please gather comprehensive information about {topic}."
    )
    research_text = research_response.message['content'][0]["text"]
    print("\n✂️ SUMMARY AGENT distilling the research\n")
    
    # Agent 2: Ask the summary agent to create a concise summary
    summary_response = summary_agent(
        f"Please create a concise summary of this research: {research_text}"
    )
    summary_text = summary_response.message['content'][0]["text"]
    
    print(summary_text)
except Exception as e:
    print(f"Error in research assistant: {str(e)}")

Multiple agents created successfully!

🔍 RESEARCH AGENT working on: generative Ai

# Comprehensive Information About Generative AI

## Definition and Overview

Generative AI refers to artificial intelligence systems capable of creating new content—including text, images, audio, video, code, and other data types—based on patterns learned from training data. Unlike discriminative AI models that classify or predict based on existing data, generative models produce novel outputs that resemble their training data but are fundamentally new creations.

## Core Technologies and Models

### Key Architectures

**Generative Adversarial Networks (GANs)**
- Introduced by Ian Goodfellow in 2014
- Consists of two neural networks: a generator and a discriminator that compete against each other
- Widely used for image generation and manipulation

**Variational Autoencoders (VAEs)**
- Probabilistic models that encode data into compressed representations
- Generate new samples by decoding from the learne

## Congrats!

You've learned how to use agents as tools in Strands Agents to create more complex agentic applications